Install Dependencies

In [ ]:
!pip install xgboost seaborn

Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

import joblib

Load Dataset

In [ ]:
df = pd.read_csv('/content/seismic_dataset_labeled_8000.csv')

print("Dataset Shape:", df.shape)
df.head()


Dataset Info

In [ ]:
df.info()
df.describe()

Feature Engineering

In [ ]:
df['stress_index'] = (1 / df['b_value']) + df['coulomb_stress']
df['seismic_intensity'] = df['magnitude'] * df['pga']
df['deformation_ratio'] = df['gps_deformation_mm'] / (df['depth_km'] + 1)
df['hydro_stress'] = df['groundwater_change'] * df['radon_level']
df['signal_quality'] = df['snr'] / (df['azimuthal_gap'] + 1)

# Advanced features
df['rolling_seismic_rate'] = df['seismicity_rate'].rolling(window=5).mean().fillna(0)
df['energy_gradient'] = df['energy_release'].diff().fillna(0)
df['depth_variation'] = df['depth_km'].diff().fillna(0)

print("Feature Engineering Completed")

b-value vs Risk

In [ ]:
plt.figure()
sns.boxplot(x='earthquake_risk', y='b_value', data=df)
plt.title("b-value vs Risk")
plt.show()

Seismicity Rate

In [ ]:
plt.figure()
sns.boxplot(x='earthquake_risk', y='seismicity_rate', data=df)
plt.title("Seismicity Rate vs Risk")
plt.show()

Magnitude

In [ ]:
plt.figure()
sns.violinplot(x='earthquake_risk', y='magnitude', data=df)
plt.title("Magnitude Distribution")
plt.show()

Depth

In [ ]:
plt.figure()
sns.boxplot(x='earthquake_risk', y='depth_km', data=df)
plt.title("Depth vs Risk")
plt.show()

PGA vs Magnitude

In [ ]:
plt.figure()
sns.scatterplot(x='magnitude', y='pga', hue='earthquake_risk', data=df)
plt.title("PGA vs Magnitude")
plt.show()

Coulomb Stress

In [ ]:
plt.figure()
sns.boxplot(x='earthquake_risk', y='coulomb_stress', data=df)
plt.title("Coulomb Stress vs Risk")
plt.show()

GPS vs Depth

In [ ]:
plt.figure()
sns.scatterplot(x='depth_km', y='gps_deformation_mm', hue='earthquake_risk', data=df)
plt.title("GPS Deformation vs Depth")
plt.show()

Radon vs Groundwater

In [ ]:
plt.figure()
sns.scatterplot(x='radon_level', y='groundwater_change', hue='earthquake_risk', data=df)
plt.title("Radon vs Groundwater")
plt.show()

Energy Release

In [ ]:
plt.figure()
sns.histplot(data=df, x='energy_release', hue='earthquake_risk', kde=True)
plt.title("Energy Distribution")
plt.show()

Inter-event Time

In [ ]:
plt.figure()
sns.boxplot(x='earthquake_risk', y='inter_event_time', data=df)
plt.title("Inter-event Time vs Risk")
plt.show()

Signal Quality

In [ ]:
plt.figure()
sns.scatterplot(x='azimuthal_gap', y='snr', hue='earthquake_risk', data=df)
plt.title("Signal Quality")
plt.show()

Correlation Heatmap

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

Pairplot

In [ ]:
sns.pairplot(df.sample(500), hue='earthquake_risk')
plt.show()

Prepare Data

In [ ]:
X = df.drop("earthquake_risk", axis=1)
y = df["earthquake_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Train Model (XGBoost)

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(X_train, y_train)

Predictions

In [ ]:
y_pred = model.predict(X_test)

Evaluation Metrics

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

Classification Report

In [ ]:
print(classification_report(y_test, y_pred))

Feature Importance

In [ ]:
importance = model.feature_importances_
features = X.columns

feat_imp = pd.Series(importance, index=features).sort_values(ascending=False)

plt.figure(figsize=(10,6))
feat_imp.plot(kind='bar')
plt.title("Feature Importance")
plt.show()

Handle Class Imbalance

In [ ]:
from imblearn.over_sampling import SMOTE
sm = SMOTE()
X_res, y_res = sm.fit_resample(X_train, y_train)

Tune XGBoost

In [ ]:
model = XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.03,
    scale_pos_weight=2
)

In [ ]:
print(classification_report(y_test, y_pred))